# CaloDiffusion: Native-cell ordering verification

**Purpose:** verify that the 368 values in a saved photon `native` shower array map to the intended detector cells, and specifically test whether native index **367** is the outermost radial bin of XML layer **12**.

This notebook does **not** generate showers, train the model, or change any run outputs. It checks the source code at the exact Git revision used by the three photon runs, reconstructs the photon geometry from the XML, tests the repository's own reshape/unreshape functions with artificial index values, and finally inspects the saved extreme shower.

## Why this matters

The extreme generated shower was reported to deposit about 4.10 GeV in native cell 367. That statement is only justified if the flat-array index is correctly mapped to the geometry. We therefore need several independent checks:

1. The checkout and relevant source files match the run's recorded revision and hashes.
2. The XML parser's geometry agrees with the geometry JSON saved with the run.
3. The repository's `XMLHandler.fill_r_a_lists()` and `GeomConverter.reshape()` implement the same angle/radial ordering.
4. `GeomConverter.reshape()` followed by `unreshape()` preserves every unique index exactly.
5. The actual event's native index 367 has the expected layer/radial metadata and deposit.

**Important:** passing these checks verifies the software mapping. It does not, by itself, establish whether the 4.10 GeV deposit is physically plausible or a model failure.

## Step 0 — Start in Google Colab

Open a fresh Colab notebook, upload this `.ipynb`, and run the cells from top to bottom. The source checkout is placed in a separate folder so it does not overwrite your existing project.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import hashlib
import json
import zipfile
import io
import inspect
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from IPython.display import display

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)

Python: 3.13.16
PyTorch: 2.11.0+cu130
NumPy: 2.4.6


## Step 1 — Check out the exact source revision

The three photon runs recorded this repository revision:

`00ebcc3534fc25eee0fe15361483833e0bca34e7`

The run manifest also recorded SHA-256 hashes for the wrapper, geometry-conversion source, and photon XML. We check those hashes so that this notebook does not silently inspect a different version of the code.

In [2]:
REPO_URL = "https://github.com/Vishal21501/gen-hep-notebooks.git"
RUN_SHA = "00ebcc3534fc25eee0fe15361483833e0bca34e7"
REPO = Path("/content/native-order-check-source")

# These hashes were recorded in the photon run manifest.
EXPECTED_HASHES = {
    "capstone/calodiffusion/calodiff_probe.py":
        "bdbd39c507134bacb1352e528260315805385809a5b9fcdf8bfae530a97f257d",
    "capstone/calodiffusion/paper_code/scripts/utils.py":
        "d34325c1fa5029fa436359b8893095e11c12bdb70a27a165b9b71c9e2164ea9e",
    "capstone/calodiffusion/paper_code/CaloChallenge/code/binning_dataset_1_photons.xml":
        "30786d54defff3d45dada4f4ecb2a4600a1622b3ca2004d40dae01774589a92f",
}

def run_git(args, env=None):
    return subprocess.run(
        ["git", "-C", str(REPO), *args],
        check=True, text=True, capture_output=True, env=env
    ).stdout.strip()

git_env = os.environ.copy()
git_env["GIT_LFS_SKIP_SMUDGE"] = "1"

if not (REPO / ".git").exists():
    if REPO.exists():
        raise RuntimeError(
            f"{REPO} already exists but is not a Git checkout. "
            "Use a fresh Colab runtime or change REPO to a new folder."
        )
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--sparse", REPO_URL, str(REPO)],
        check=True, env=git_env
    )
    run_git(["sparse-checkout", "set", "capstone/calodiffusion"], env=git_env)

# Fetch the pinned commit even if the default branch has moved on.
run_git(["fetch", "--depth", "1", "origin", RUN_SHA], env=git_env)
current_sha = run_git(["rev-parse", "HEAD"])
if current_sha != RUN_SHA:
    dirty = run_git(["status", "--porcelain"])
    if dirty:
        raise RuntimeError(
            "The source checkout has local modifications; refusing to switch revisions. "
            "Use a fresh Colab runtime."
        )
    run_git(["checkout", "--detach", RUN_SHA])

actual_sha = run_git(["rev-parse", "HEAD"])
assert actual_sha == RUN_SHA, (actual_sha, RUN_SHA)

PROJECT = REPO / "capstone" / "calodiffusion"
PAPER_CODE = PROJECT / "paper_code"
SCRIPTS = PAPER_CODE / "scripts"
XML_PATH = PAPER_CODE / "CaloChallenge" / "code" / "binning_dataset_1_photons.xml"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Pinned Git revision:", actual_sha)
for rel, expected in EXPECTED_HASHES.items():
    path = REPO / rel
    if not path.is_file():
        raise FileNotFoundError(f"Required source file missing: {path}")
    actual = sha256_file(path)
    print(f"{rel}\n  SHA256: {actual}\n  MATCH: {actual == expected}")
    assert actual == expected, f"Source hash mismatch: {rel}"

print("\nPASS: repository revision and all three source hashes match the run manifest.")

Pinned Git revision: 00ebcc3534fc25eee0fe15361483833e0bca34e7
capstone/calodiffusion/calodiff_probe.py
  SHA256: bdbd39c507134bacb1352e528260315805385809a5b9fcdf8bfae530a97f257d
  MATCH: True
capstone/calodiffusion/paper_code/scripts/utils.py
  SHA256: d34325c1fa5029fa436359b8893095e11c12bdb70a27a165b9b71c9e2164ea9e
  MATCH: True
capstone/calodiffusion/paper_code/CaloChallenge/code/binning_dataset_1_photons.xml
  SHA256: 30786d54defff3d45dada4f4ecb2a4600a1622b3ca2004d40dae01774589a92f
  MATCH: True

PASS: repository revision and all three source hashes match the run manifest.


## Step 2 — Import the repository's actual geometry code

`utils.py` imports `XMLHandler` from `CaloChallenge/code/XMLHandler.py` and defines `GeomConverter`. The source expects the working directory to be `paper_code/scripts`, with `paper_code` on the Python path. We set that explicitly before importing.

We will use the repository implementation, not a replacement implementation written for this audit.

In [3]:
# Make the repository's relative imports resolve as expected.
os.chdir(SCRIPTS)
sys.path.insert(0, str(PAPER_CODE))
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(PROJECT))

import utils as calo_utils

assert Path(calo_utils.__file__).resolve() == (SCRIPTS / "utils.py").resolve(), (
    f"Imported the wrong utils module: {calo_utils.__file__}"
)

assert hasattr(calo_utils, "XMLHandler")
assert hasattr(calo_utils, "GeomConverter")

bins = calo_utils.XMLHandler("photon", str(XML_PATH))
geom_converter = calo_utils.GeomConverter(bins)

print("Imported utils from:", calo_utils.__file__)
print("XMLHandler total bins:", bins.GetTotalNumberOfBins())
print("XMLHandler bin edges:", bins.GetBinEdges())
print("GeomConverter layer boundaries:", geom_converter.layer_boundaries)
print("GeomConverter angular bins:", geom_converter.lay_alphas)
print("GeomConverter layer count:", geom_converter.num_layers)

assert int(bins.GetTotalNumberOfBins()) == 368
assert np.array_equal(
    np.asarray(geom_converter.layer_boundaries, dtype=int),
    np.array([0, 8, 168, 358, 363, 368])
)
assert list(map(int, geom_converter.lay_alphas)) == [1, 10, 10, 1, 1]
print("\nPASS: XMLHandler and GeomConverter report 368 native cells in five active layers.")

Imported utils from: /content/native-order-check-source/capstone/calodiffusion/paper_code/scripts/utils.py
XMLHandler total bins: 368
XMLHandler bin edges: [0, 8, 168, 358, 363, 363, 363, 363, 363, 363, 363, 363, 363, 368, 368, 368, 368, 368, 368, 368, 368, 368, 368, 368, 368]
GeomConverter layer boundaries: [  0   8 168 358 363 368]
GeomConverter angular bins: [1, 10, 10, 1, 1]
GeomConverter layer count: 5

PASS: XMLHandler and GeomConverter report 368 native cells in five active layers.


## Step 3 — Inspect the source functions that define the ordering

There are two relevant operations:

- `XMLHandler.fill_r_a_lists()` loops over angular bins and, inside that loop, radial bins.
- `GeomConverter.reshape()` slices each layer from the flat vector and reshapes it to `(showers, angular_bins, radial_bins)`.

With NumPy/PyTorch's standard row-major reshape, the radial index changes fastest. The flat index within a layer is therefore:

\[
 i = i_{\rm layer\ start} + i_{\rm angle}\,N_r + i_{\rm radial}.
\]

The cell below prints the actual source methods from the pinned checkout.

In [4]:
print("===== XMLHandler.fill_r_a_lists =====")
print(inspect.getsource(calo_utils.XMLHandler.fill_r_a_lists))

print("\n===== GeomConverter.reshape =====")
print(inspect.getsource(calo_utils.GeomConverter.reshape))

print("\n===== GeomConverter.unreshape =====")
print(inspect.getsource(calo_utils.GeomConverter.unreshape))

print("\n===== NNConverter.dec (decoder path) =====")
print(inspect.getsource(calo_utils.NNConverter.dec))

===== XMLHandler.fill_r_a_lists =====
    def fill_r_a_lists(self, layer):
        no_of_rbins = self.r_bins[layer]
        list_mid_values = self.r_midvalue[layer]
        list_a_values = self.alphaListPerLayer[layer]
        r_list = []
        a_list = []
        actual_no_alpha_bins = self.nBinAlphaPerlayer[layer][0]
        for j0 in range(0, actual_no_alpha_bins):
            for i0 in range(0, no_of_rbins):
                r_list.append(list_mid_values[i0])
                a_list.append(list_a_values[i0][j0])
        return r_list, a_list


===== GeomConverter.reshape =====
    def reshape(self, raw_shower):
        #convert to jagged array each of shape (N_shower, N_alpha, N_R)
        shower_reshape = []
        for idx in range(len(self.layer_boundaries)-1):
            data_reshaped = raw_shower[:,self.layer_boundaries[idx]:self.layer_boundaries[idx+1]].reshape(raw_shower.shape[0], int(self.lay_alphas[idx]), -1)
            shower_reshape.append(data_reshaped)
        return

## Step 4 — Upload the saved photon run and compare its geometry with the XML

Upload the ZIP for **seed 20261021**:

`validated_calo_smoke_20261024T161135564483Z_fidelity_20261024T163205369907Z.zip`

The ZIP contains `manifest.json`, `photon_geometry.json`, and the saved native shower array. This cell verifies the seed and source revision, then compares the JSON geometry with the XML parser's geometry.

In [5]:
from google.colab import files

uploaded = files.upload()
zip_uploads = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_uploads) != 1:
    raise ValueError("Upload exactly one run ZIP: the photon run with seed 20261021.")

run_zip_name = zip_uploads[0]
run_zip_bytes = uploaded[run_zip_name]

with zipfile.ZipFile(io.BytesIO(run_zip_bytes)) as zf:
    names = zf.namelist()

    def unique_member(suffix):
        found = [name for name in names if name.endswith(suffix)]
        if len(found) != 1:
            raise ValueError(f"Expected one ZIP member ending in {suffix!r}; found {len(found)}")
        return found[0]

    manifest = json.loads(zf.read(unique_member("manifest.json")))
    saved_geometry = json.loads(zf.read(unique_member("photon_geometry.json")))

    npz_member = unique_member("arrays/photon_65.536GeV_generator_run0.npz")
    with np.load(io.BytesIO(zf.read(npz_member)), allow_pickle=False) as npz:
        native_array = npz["native"].copy()
        event_energies = npz["energy"].copy()
        event_ids = npz["event_id"].astype(str).copy()

assert manifest["config"]["seed"] == 20261021
assert manifest["revision"] == RUN_SHA
assert manifest["source_sha256"]["paper_code/scripts/utils.py"] == EXPECTED_HASHES[
    "capstone/calodiffusion/paper_code/scripts/utils.py"
]
assert manifest["source_sha256"]["paper_code/CaloChallenge/code/binning_dataset_1_photons.xml"] == EXPECTED_HASHES[
    "capstone/calodiffusion/paper_code/CaloChallenge/code/binning_dataset_1_photons.xml"
]

print("Run tag:", manifest["run_tag"])
print("Seed:", manifest["config"]["seed"])
print("Git revision:", manifest["revision"])
print("Native shower array shape:", native_array.shape)
print("Saved geometry layer IDs:", saved_geometry["layer_ids"])
print("Saved geometry angular bins:", saved_geometry["n_angles"])
print("PASS: uploaded ZIP belongs to the expected photon run.")

Saving validated_calo_smoke_20261024T161135564483Z_fidelity_20261024T163205369907Z.zip to validated_calo_smoke_20261024T161135564483Z_fidelity_20261024T163205369907Z.zip
Run tag: validated_calo_smoke_20261024T161135564483Z
Seed: 20261021
Git revision: 00ebcc3534fc25eee0fe15361483833e0bca34e7
Native shower array shape: (300, 368)
Saved geometry layer IDs: [0, 1, 2, 3, 12]
Saved geometry angular bins: [1, 10, 10, 1, 1]
PASS: uploaded ZIP belongs to the expected photon run.


## Step 5 — Independently reconstruct the map from the photon XML

The XML parser retains the radial edges and number of angular bins for each active layer. We compare these directly with the geometry JSON saved by the run.

Then we build the flat index map using the ordering implied by the source code: **layer → angle → radial bin**.

In [8]:
xml_root = ET.parse(XML_PATH).getroot()
xml_particle = next(
    (node for node in xml_root if node.attrib.get("name") == "photon"),
    None
)
if xml_particle is None:
    raise ValueError("Could not find the photon entry in the XML geometry.")

xml_layers = []
for layer_node in xml_particle:
    edges = np.asarray(
        [float(x) for x in layer_node.attrib["r_edges"].split(",")],
        dtype=float
    )
    n_angles = int(layer_node.attrib["n_bin_alpha"])
    # Only include active layers that have both angular and radial divisions
    if n_angles > 0 and len(edges) > 1:
        xml_layers.append({
            "layer_id": int(layer_node.attrib["id"]),
            "n_angles": n_angles,
            "radial_edges": edges,
        })

assert len(xml_layers) == len(saved_geometry["layer_ids"])
assert [x["layer_id"] for x in xml_layers] == saved_geometry["layer_ids"]
assert [x["n_angles"] for x in xml_layers] == saved_geometry["n_angles"]

for xml_layer, saved_edges in zip(xml_layers, saved_geometry["radial_edges"]):
    np.testing.assert_allclose(
        xml_layer["radial_edges"], np.asarray(saved_edges, dtype=float),
        rtol=0, atol=1e-12
    )

# Filter the XMLHandler parsed list to only include active layer IDs
active_indices = saved_geometry["layer_ids"]
parsed_active_alphas = [int(bins.a_bins[i]) for i in active_indices]
parsed_active_r_edges = [bins.r_edges[i] for i in active_indices]

assert parsed_active_alphas == list(map(int, saved_geometry["n_angles"]))
for parsed_edges, saved_edges in zip(parsed_active_r_edges, saved_geometry["radial_edges"]):
    np.testing.assert_allclose(parsed_edges, saved_edges, rtol=0, atol=1e-12)

cell_rows = []
layer_rows = []
offset = 0
for layer_ordinal, layer in enumerate(xml_layers):
    edges = layer["radial_edges"]
    n_radial = len(edges) - 1
    n_angles = layer["n_angles"]
    start = offset
    stop = start + n_angles * n_radial

    layer_rows.append({
        "layer_ordinal": layer_ordinal,
        "layer_id": layer["layer_id"],
        "n_angles": n_angles,
        "n_radial": n_radial,
        "native_start": start,
        "native_stop_exclusive": stop,
    })

    for angle_index in range(n_angles):
        for radial_bin in range(n_radial):
            idx = start + angle_index * n_radial + radial_bin
            cell_rows.append({
                "native_index": idx,
                "layer_ordinal": layer_ordinal,
                "layer_id": layer["layer_id"],
                "angle_index": angle_index,
                "radial_bin": radial_bin,
                "r_low_XML": edges[radial_bin],
                "r_high_XML": edges[radial_bin + 1],
                "r_center_XML": 0.5 * (edges[radial_bin] + edges[radial_bin + 1]),
            })
    offset = stop

cell_map = pd.DataFrame(cell_rows)
layer_map = pd.DataFrame(layer_rows)

expected_boundaries = np.asarray(
    [0] + list(np.cumsum([
        row["n_angles"] * row["n_radial"] for row in layer_rows
    ])), dtype=int
)

assert len(cell_map) == 368
assert offset == 368
np.testing.assert_array_equal(expected_boundaries, geom_converter.layer_boundaries)
np.testing.assert_array_equal(
    np.asarray([row["n_angles"] for row in layer_rows]),
    np.asarray(geom_converter.lay_alphas)
)

print("XML geometry and saved JSON agree.")
print("XMLHandler radial edges and angular counts agree with JSON.")
print("Native-cell count:", len(cell_map))
display(layer_map)
print("Last eight native cells:")
display(cell_map.tail(8))

XML geometry and saved JSON agree.
XMLHandler radial edges and angular counts agree with JSON.
Native-cell count: 368


,layer_ordinal,layer_id,n_angles,n_radial,native_start,native_stop_exclusive
0,0,0,1,8,0,8
1,1,1,10,16,8,168
2,2,2,10,19,168,358
3,3,3,1,5,358,363
4,4,12,1,5,363,368


Last eight native cells:


,native_index,layer_ordinal,layer_id,angle_index,radial_bin,r_low_XML,r_high_XML,r_center_XML
360,360,3,3,0,2,100.0,200.0,150.0
361,361,3,3,0,3,200.0,400.0,300.0
362,362,3,3,0,4,400.0,600.0,500.0
363,363,4,12,0,0,0.0,100.0,50.0
364,364,4,12,0,1,100.0,200.0,150.0
365,365,4,12,0,2,200.0,400.0,300.0
366,366,4,12,0,3,400.0,1000.0,700.0
367,367,4,12,0,4,1000.0,2000.0,1500.0


## Step 6 — Test the repository's reshape/unreshape ordering with unique sentinels

This is the direct software test. Instead of using real energy deposits, we create a fake shower whose cell values are exactly `0, 1, 2, ..., 367`. Because every value is unique, we can see exactly which flat index goes to each layer/angle/radial position.

We then:

1. call the repository's `GeomConverter.reshape()`;
2. compare every resulting layer tensor with the expected index matrix;
3. call `GeomConverter.unreshape()`;
4. require exact equality with the original flat vector.

This tests the actual conversion implementation, not just our own formula.

In [9]:
sentinel = torch.arange(368, dtype=torch.float32).reshape(1, 368)
layer_tensors = geom_converter.reshape(sentinel)

assert len(layer_tensors) == len(layer_rows)

sentinel_checks = []
for j, row in enumerate(layer_rows):
    start = int(row["native_start"])
    stop = int(row["native_stop_exclusive"])
    n_angles = int(row["n_angles"])
    n_radial = int(row["n_radial"])

    observed = layer_tensors[j].detach().cpu().numpy()
    expected = np.arange(start, stop, dtype=np.float32).reshape(
        1, n_angles, n_radial
    )

    assert observed.shape == expected.shape, (
        f"Layer {row['layer_id']} shape mismatch: "
        f"observed {observed.shape}, expected {expected.shape}"
    )
    np.testing.assert_array_equal(observed, expected)

    sentinel_checks.append({
        "layer_id": row["layer_id"],
        "expected_shape": str(expected.shape),
        "first_native_index": start,
        "last_native_index": stop - 1,
        "result": "PASS",
    })

# Test the reverse operation, which is used when returning layer arrays to native order.
restored = geom_converter.unreshape(layer_tensors)
assert tuple(restored.shape) == (1, 368)
assert torch.equal(restored.cpu(), sentinel.cpu()), (
    "GeomConverter.unreshape(GeomConverter.reshape(x)) did not reproduce x exactly."
)

print("Per-layer reshape checks:")
display(pd.DataFrame(sentinel_checks))
print("PASS: reshape followed by unreshape reproduces all 368 indices exactly.")

Per-layer reshape checks:


,layer_id,expected_shape,first_native_index,last_native_index,result
0,0,"(1, 1, 8)",0,7,PASS
1,1,"(1, 10, 16)",8,167,PASS
2,2,"(1, 10, 19)",168,357,PASS
3,3,"(1, 1, 5)",358,362,PASS
4,12,"(1, 1, 5)",363,367,PASS


PASS: reshape followed by unreshape reproduces all 368 indices exactly.


## Step 7 — Verify the decoder uses the native unreshape operation

`NNConverter.dec()` decodes each layer and calls `self.gc.unreshape(out)` before returning the flat native representation. The source shown below is the decoder path used by the geometry conversion module. The `unreshape()` round-trip test above therefore checks the final reassembly convention it relies on.

In [10]:
decoder_source = inspect.getsource(calo_utils.NNConverter.dec)
print(decoder_source)
assert "self.gc.unreshape(out)" in decoder_source
print("PASS: NNConverter.dec() returns through GeomConverter.unreshape().")

    def dec(self, x):
        out = []
        x = torch.squeeze(x, dim=1)
        for i in range(self.gc.num_layers):
            o = self.decs[i](x[:,i])

            if(self.gc.lay_alphas is not None):
                if(self.gc.lay_alphas[i]  == 1):
                    #Only works for converting 1 alpha bin into multiple, ok for dataset1 but maybe should generalize
                    o = torch.sum(o, dim = -2, keepdim = True)
                elif(self.gc.lay_alphas[i]  != self.gc.alpha_out):
                    print("Num alpha bins for layer %i is %i. Don't know how to handle" % (i, self.gc.lay_alphas[i]))
                    exit(1)
            out.append(o)
        out = self.gc.unreshape(out)
        return out

PASS: NNConverter.dec() returns through GeomConverter.unreshape().


## Step 8 — Find and inspect the actual extreme shower

Now that the source ordering and XML geometry agree, we inspect the saved event:

`gen:photon:65.536GeV:0:161` (seed `20261021`, generator run 0).

The event is not regenerated. We read its saved native array from the uploaded ZIP and attach the verified geometry metadata to each value.

In [11]:
TARGET_EVENT_ID = "gen:photon:65.536GeV:0:161"
CUT_GEV = 0.01  # Same 10 MeV analysis cutoff used in the photon follow-up.

matches = np.flatnonzero(event_ids == TARGET_EVENT_ID)
if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one {TARGET_EVENT_ID!r}; found {len(matches)}. "
        "Check that the uploaded ZIP is the seed 20261021 run."
    )

event_row = int(matches[0])
native_raw = np.asarray(native_array[event_row], dtype=float)
incident_energy = float(event_energies[event_row])

assert native_raw.shape == (368,)
assert np.isclose(incident_energy, 65.536)

cell_audit = cell_map.copy()
cell_audit["energy_raw_GeV"] = native_raw
cell_audit["energy_kept_GeV"] = np.where(
    native_raw >= CUT_GEV, native_raw, 0.0
)
kept_total = float(cell_audit["energy_kept_GeV"].sum())
raw_total = float(native_raw.sum())
cell_audit["fraction_of_kept_energy"] = (
    cell_audit["energy_kept_GeV"] / kept_total if kept_total > 0 else 0.0
)

print("Event:", TARGET_EVENT_ID)
print(f"Incident energy: {incident_energy:.3f} GeV")
print(f"Raw deposited energy: {raw_total:.6f} GeV")
print(f"Deposited energy retained above 10 MeV: {kept_total:.6f} GeV")
print(f"Retained response: {kept_total / incident_energy:.6f}")
print("Native array length:", len(native_raw))

print("\nAll cells in the final XML layer:")
final_layer_id = int(xml_layers[-1]["layer_id"])
final_layer_cells = cell_audit[cell_audit["layer_id"] == final_layer_id]
display(final_layer_cells[[
    "native_index", "layer_id", "angle_index", "radial_bin",
    "r_low_XML", "r_high_XML", "energy_raw_GeV", "energy_kept_GeV",
    "fraction_of_kept_energy"
]])

print("\nNative cell 367:")
display(cell_audit[cell_audit["native_index"] == 367][[
    "native_index", "layer_ordinal", "layer_id", "angle_index",
    "radial_bin", "r_low_XML", "r_high_XML",
    "energy_raw_GeV", "energy_kept_GeV"
]])

Event: gen:photon:65.536GeV:0:161
Incident energy: 65.536 GeV
Raw deposited energy: 59.799182 GeV
Deposited energy retained above 10 MeV: 59.552852 GeV
Retained response: 0.908704
Native array length: 368

All cells in the final XML layer:


,native_index,layer_id,angle_index,radial_bin,r_low_XML,r_high_XML,energy_raw_GeV,energy_kept_GeV,fraction_of_kept_energy
363,363,12,0,0,0.0,100.0,6.033397,6.033397,0.101312
364,364,12,0,1,100.0,200.0,1.530282,1.530282,0.025696
365,365,12,0,2,200.0,400.0,1.819340,1.819340,0.030550
366,366,12,0,3,400.0,1000.0,1.516879,1.516879,0.025471
367,367,12,0,4,1000.0,2000.0,4.097897,4.097897,0.068811



Native cell 367:


,native_index,layer_ordinal,layer_id,angle_index,radial_bin,r_low_XML,r_high_XML,energy_raw_GeV,energy_kept_GeV
367,367,4,12,0,4,1000.0,2000.0,4.097897,4.097897


## Step 9 — Final assertions and save an audit table

The final cell collects the checks. If all assertions pass, the notebook has verified the mapping used to interpret this saved event.

In [12]:
outer_row = cell_audit.loc[cell_audit["native_index"] == 367].iloc[0]
final_layer_energy = float(final_layer_cells["energy_kept_GeV"].sum())
outer_cell_energy = float(outer_row["energy_kept_GeV"])

# Exact geometry expectations for the final native cell.
assert int(outer_row["layer_id"]) == 12
assert int(outer_row["layer_ordinal"]) == 4
assert int(outer_row["angle_index"]) == 0
assert int(outer_row["radial_bin"]) == 4
assert np.isclose(float(outer_row["r_low_XML"]), 1000.0)
assert np.isclose(float(outer_row["r_high_XML"]), 2000.0)
assert np.isclose(outer_cell_energy, 4.098, atol=0.02)

summary = pd.DataFrame([{
    "source_revision": actual_sha,
    "seed": manifest["config"]["seed"],
    "native_cell_count": len(cell_map),
    "event_id": TARGET_EVENT_ID,
    "native_index": 367,
    "xml_layer_id": int(outer_row["layer_id"]),
    "layer_ordinal": int(outer_row["layer_ordinal"]),
    "angle_index": int(outer_row["angle_index"]),
    "radial_bin": int(outer_row["radial_bin"]),
    "r_low_XML": float(outer_row["r_low_XML"]),
    "r_high_XML": float(outer_row["r_high_XML"]),
    "outer_cell_energy_GeV_after_cut": outer_cell_energy,
    "final_layer_energy_GeV_after_cut": final_layer_energy,
    "total_energy_GeV_after_cut": kept_total,
}])

display(summary)

OUTPUT_DIR = Path("/content/native_cell_order_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cell_audit.to_csv(OUTPUT_DIR / "event_161_native_cell_map.csv", index=False)
summary.to_csv(OUTPUT_DIR / "native_cell_order_verification_summary.csv", index=False)

print("\nALL CHECKS PASSED.")
print("Interpretation: native index 367 maps to XML layer 12, angle 0, radial bin 4,")
print("with radial edges 1000–2000 XML units in this pinned source/geometry.")
print("The saved event has approximately 4.10 GeV in that cell after the 10 MeV cutoff.")
print("CSV audit files saved in:", OUTPUT_DIR)

,source_revision,seed,native_cell_count,event_id,native_index,xml_layer_id,layer_ordinal,angle_index,radial_bin,r_low_XML,r_high_XML,outer_cell_energy_GeV_after_cut,final_layer_energy_GeV_after_cut,total_energy_GeV_after_cut
0,00ebcc3534fc25eee0fe15361483833e0bca34e7,20261021,368,gen:photon:65.536GeV:0:161,367,12,4,0,4,1000.0,2000.0,4.097897,14.997795,59.552852



ALL CHECKS PASSED.
Interpretation: native index 367 maps to XML layer 12, angle 0, radial bin 4,
with radial edges 1000–2000 XML units in this pinned source/geometry.
The saved event has approximately 4.10 GeV in that cell after the 10 MeV cutoff.
CSV audit files saved in: /content/native_cell_order_audit


# **Safer Check for XML Handler - To ensure correct native ordering [Just another check]**

In [13]:

# ============================================================
# STEP 5: VERIFY XML, SAVED GEOMETRY AND NATIVE CELL ORDERING
# ============================================================

# Required existing variables:
# XML_PATH, saved_geometry, bins, geom_converter
# Imports: numpy as np, pandas as pd, torch
#          xml.etree.ElementTree as ET

# ------------------------------------------------------------
# PART A: Read photon geometry directly from the XML
# ------------------------------------------------------------

xml_root = ET.parse(XML_PATH).getroot()

xml_particle = next(
    (node for node in xml_root
     if node.attrib.get("name") == "photon"),
    None
)

if xml_particle is None:
    raise ValueError("Could not find photon entry in XML geometry.")

xml_nodes = list(xml_particle)

xml_layers = []

for xml_position, layer_node in enumerate(xml_nodes):

    edges = np.asarray(
        [float(x) for x in layer_node.attrib["r_edges"].split(",")],
        dtype=float
    )

    n_angles = int(layer_node.attrib["n_bin_alpha"])
    layer_id = int(layer_node.attrib["id"])

    # Include only active detector layers.
    if n_angles > 0 and len(edges) > 1:

        xml_layers.append({
            "xml_position": xml_position,
            "layer_id": layer_id,
            "n_angles": n_angles,
            "radial_edges": edges
        })


# ------------------------------------------------------------
# PART B: Compare XML geometry with saved JSON
# ------------------------------------------------------------

saved_ids = list(map(int, saved_geometry["layer_ids"]))
saved_angles = list(map(int, saved_geometry["n_angles"]))

xml_ids = [x["layer_id"] for x in xml_layers]
xml_angles = [x["n_angles"] for x in xml_layers]

assert xml_ids == saved_ids, (
    "XML layer IDs do not match saved geometry."
)

assert xml_angles == saved_angles, (
    "XML angular-bin counts do not match saved geometry."
)

assert len(xml_layers) == len(saved_geometry["radial_edges"])

for xml_layer, saved_edges in zip(
    xml_layers, saved_geometry["radial_edges"]
):
    np.testing.assert_allclose(
        xml_layer["radial_edges"],
        np.asarray(saved_edges, dtype=float),
        rtol=0,
        atol=1e-12
    )

print("PASS: XML geometry matches saved JSON.")


# ------------------------------------------------------------
# PART C: Compare XML geometry with XMLHandler
# ------------------------------------------------------------

# XMLHandler stores entries in XML document order.
# GetRelevantLayers() returns the active XML layer IDs.

xml_positions = [
    x["xml_position"] for x in xml_layers
]

handler_active_ids = list(
    map(int, bins.GetRelevantLayers())
)

assert handler_active_ids == xml_ids, (
    "XMLHandler active layer IDs do not match XML."
)

# For this XML, IDs also correspond to document positions.
# Verify this explicitly because GeomConverter uses these
# values when selecting the active radial-edge lists.

assert handler_active_ids == xml_positions, (
    "XML layer IDs are not identical to XML positions. "
    "Investigate the converter indexing before proceeding."
)

parsed_active_alphas = [
    int(bins.a_bins[pos]) for pos in xml_positions
]

parsed_active_edges = [
    np.asarray(bins.r_edges[pos], dtype=float)
    for pos in xml_positions
]

assert parsed_active_alphas == saved_angles

for parsed_edges, saved_edges in zip(
    parsed_active_edges, saved_geometry["radial_edges"]
):
    np.testing.assert_allclose(
        parsed_edges,
        np.asarray(saved_edges, dtype=float),
        rtol=0,
        atol=1e-12
    )

print("PASS: XMLHandler angular bins match.")
print("PASS: XMLHandler radial edges match.")


# ------------------------------------------------------------
# PART D: Reconstruct the native cell-index map
# ------------------------------------------------------------

cell_rows = []
layer_rows = []

offset = 0

for layer_ordinal, layer in enumerate(xml_layers):

    edges = layer["radial_edges"]

    n_radial = len(edges) - 1
    n_angles = layer["n_angles"]

    start = offset
    stop = start + n_angles * n_radial

    layer_rows.append({
        "layer_ordinal": layer_ordinal,
        "layer_id": layer["layer_id"],
        "n_angles": n_angles,
        "n_radial": n_radial,
        "native_start": start,
        "native_stop_exclusive": stop
    })

    # Expected order:
    # Layer -> Angular bin -> Radial bin
    # Radial bin changes fastest.

    for angle_index in range(n_angles):

        for radial_bin in range(n_radial):

            idx = (
                start
                + angle_index * n_radial
                + radial_bin
            )

            cell_rows.append({
                "native_index": idx,
                "layer_ordinal": layer_ordinal,
                "layer_id": layer["layer_id"],
                "angle_index": angle_index,
                "radial_bin": radial_bin,
                "r_low_XML": edges[radial_bin],
                "r_high_XML": edges[radial_bin + 1],
                "r_center_XML": (
                    edges[radial_bin] + edges[radial_bin + 1]
                ) / 2
            })

    offset = stop


cell_map = pd.DataFrame(cell_rows)
layer_map = pd.DataFrame(layer_rows)

# Calculate expected cumulative native boundaries.
expected_boundaries = np.asarray(
    [0] + list(np.cumsum(
        [row["n_angles"] * row["n_radial"]
         for row in layer_rows]
    )),
    dtype=int
)

# Compare with actual CaloDiffusion converter.
np.testing.assert_array_equal(
    expected_boundaries,
    np.asarray(geom_converter.layer_boundaries, dtype=int)
)

np.testing.assert_array_equal(
    np.asarray([x["n_angles"] for x in layer_rows]),
    np.asarray(geom_converter.lay_alphas, dtype=int)
)

assert len(cell_map) == 368
assert offset == 368
assert bins.GetTotalNumberOfBins() == 368
assert geom_converter.num_layers == len(xml_layers)

print("\nPASS: Native cell count =", len(cell_map))
print("PASS: Layer boundaries match GeomConverter.")
print("PASS: Angular-bin counts match GeomConverter.")

print("\nLayer summary:")
display(layer_map)

print("\nLast eight native cells:")
display(cell_map.tail(8))


# ------------------------------------------------------------
# PART E: ARTIFICIAL-NUMBER (SENTINEL) TEST
# ------------------------------------------------------------

# Create a fake shower with unique values:
# Cell 0 contains 0, cell 1 contains 1, ... cell 367 contains 367.

sentinel = torch.arange(
    368, dtype=torch.float32
).reshape(1, 368)

# Apply the ACTUAL repository reshape function.
layer_tensors = geom_converter.reshape(sentinel)

assert len(layer_tensors) == len(layer_rows)

sentinel_results = []

for j, row in enumerate(layer_rows):

    start = int(row["native_start"])
    stop = int(row["native_stop_exclusive"])

    n_angles = int(row["n_angles"])
    n_radial = int(row["n_radial"])

    # Expected arrangement for this detector layer.
    expected = np.arange(
        start, stop, dtype=np.float32
    ).reshape(1, n_angles, n_radial)

    # Actual arrangement produced by CaloDiffusion.
    observed = (
        layer_tensors[j]
        .detach()
        .cpu()
        .numpy()
    )

    assert observed.shape == expected.shape, (
        f"Shape mismatch in layer {row['layer_id']}"
    )

    np.testing.assert_array_equal(
        observed, expected
    )

    sentinel_results.append({
        "layer_id": row["layer_id"],
        "shape": str(observed.shape),
        "first_index": start,
        "last_index": stop - 1,
        "result": "PASS"
    })

print("\nArtificial-number reshape test:")
display(pd.DataFrame(sentinel_results))


# ------------------------------------------------------------
# PART F: REVERSE THE OPERATION
# ------------------------------------------------------------

# Convert the layer tensors back to the original flat array.
restored = geom_converter.unreshape(layer_tensors)

assert tuple(restored.shape) == (1, 368)

assert torch.equal(
    restored.cpu(),
    sentinel.cpu()
), "Reshape/unreshape round-trip failed!"

print("PASS: Reshape -> unreshape reproduces all 368 values.")


# ------------------------------------------------------------
# PART G: VERIFY THE SPECIFIC OUTERMOST CELL
# ------------------------------------------------------------

outer_cell = cell_map.loc[
    cell_map["native_index"] == 367
].iloc[0]

assert int(outer_cell["layer_id"]) == 12
assert int(outer_cell["layer_ordinal"]) == 4
assert int(outer_cell["angle_index"]) == 0
assert int(outer_cell["radial_bin"]) == 4

assert np.isclose(outer_cell["r_low_XML"], 1000.0)
assert np.isclose(outer_cell["r_high_XML"], 2000.0)

print("\nVERIFIED NATIVE CELL 367:")
display(outer_cell.to_frame("value"))

print("\n" + "=" * 55)
print("ALL GEOMETRY AND ORDERING TESTS PASSED")
print("=" * 55)


PASS: XML geometry matches saved JSON.
PASS: XMLHandler angular bins match.
PASS: XMLHandler radial edges match.

PASS: Native cell count = 368
PASS: Layer boundaries match GeomConverter.
PASS: Angular-bin counts match GeomConverter.

Layer summary:


,layer_ordinal,layer_id,n_angles,n_radial,native_start,native_stop_exclusive
0,0,0,1,8,0,8
1,1,1,10,16,8,168
2,2,2,10,19,168,358
3,3,3,1,5,358,363
4,4,12,1,5,363,368



Last eight native cells:


,native_index,layer_ordinal,layer_id,angle_index,radial_bin,r_low_XML,r_high_XML,r_center_XML
360,360,3,3,0,2,100.0,200.0,150.0
361,361,3,3,0,3,200.0,400.0,300.0
362,362,3,3,0,4,400.0,600.0,500.0
363,363,4,12,0,0,0.0,100.0,50.0
364,364,4,12,0,1,100.0,200.0,150.0
365,365,4,12,0,2,200.0,400.0,300.0
366,366,4,12,0,3,400.0,1000.0,700.0
367,367,4,12,0,4,1000.0,2000.0,1500.0



Artificial-number reshape test:


,layer_id,shape,first_index,last_index,result
0,0,"(1, 1, 8)",0,7,PASS
1,1,"(1, 10, 16)",8,167,PASS
2,2,"(1, 10, 19)",168,357,PASS
3,3,"(1, 1, 5)",358,362,PASS
4,12,"(1, 1, 5)",363,367,PASS


PASS: Reshape -> unreshape reproduces all 368 values.

VERIFIED NATIVE CELL 367:


,value
native_index,367.0
layer_ordinal,4.0
layer_id,12.0
angle_index,0.0
radial_bin,4.0
r_low_XML,1000.0
r_high_XML,2000.0
r_center_XML,1500.0



ALL GEOMETRY AND ORDERING TESTS PASSED


# How to interpret the outcome

### If all checks pass

We have verified the **software mapping**: native index 367 corresponds to XML layer 12, angular bin 0, radial bin 4, spanning 1000–2000 XML radial units. The saved event has about 4.10 GeV in that cell after the 10 MeV cutoff. Continue by investigating whether this is a plausible rare shower or a generator-related spatial tail.

### If the XML/JSON comparison fails

Stop before interpreting the radial result. The geometry saved with the run and the checked-out XML do not agree. Confirm the run revision and source files before changing any analysis.

### If the sentinel reshape test fails

Stop. The expected flattening formula does not match the actual converter behaviour in this checkout. Do not use the current cell map or radial RMS interpretation until the discrepancy is understood.

### What this does not prove

This notebook does not prove that a 4.10 GeV deposit in the outermost radial cell is physically impossible. It verifies where the saved value is placed by the pinned software geometry and verifies that the native flattening convention is internally consistent. Physical plausibility must be assessed separately against the reference population, detector setup, and simulation conventions.